In [2]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

In [3]:
def edit_ls_code_column(x):
    value=x.split('_')
    if len(value)>3:
        route_value="_".join(value[:-1])
    else:
        route_value="_".join(value)
    return route_value


In [4]:

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:
detail_df_stops=pd.read_excel('details_project_od_excel_UTA.xlsx',sheet_name='STOPS')
detail_df_xfers=pd.read_excel('details_project_od_excel_UTA.xlsx',sheet_name='XFERS')

In [6]:

wkend_overall_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkEND-RAIL')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkEND-RailTotal')

df=pd.read_csv("elvisutaobweekday2024_export_odbc.csv")

wkday_overall_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkDAY-RAIL')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('UTA_SL_CR.xlsx',sheet_name='WkDAY-RailTotal')



In [7]:
wkend_overall_df['STATION_ID_SPLITTED']=wkend_overall_df['STATION_ID'].apply(lambda x: str(x).split('_')[-1])

In [8]:
wkday_overall_df['STATION_ID_SPLITTED']=wkday_overall_df['STATION_ID'].apply(lambda x: str(x).split('_')[-1])

In [9]:
stop_on_clntid=['stoponclntid']
stop_on_clntid=check_all_characters_present(df,stop_on_clntid)
stop_on_clntid

['STOP_ON_CLNTID']

In [10]:
df['STATION_ID_SPLITTED']=df[stop_on_clntid[0]].apply(lambda x:str(x).split('_')[-1])
df[['STATION_ID_SPLITTED',stop_on_clntid[0]]].head()

,STATION_ID_SPLITTED,STOP_ON_CLNTID
0,nan,NaN
1,nan,NaN
2,14114,UTA_1_209_01_14114
3,14596,UTA_1_200_00_14596
4,15584,UTA_1_200_01_15584


In [11]:
'UTA_1_703_19171'.split('_')[-1]

'19171'

In [12]:
wkday_route_df['ROUTE_TOTAL'] = pd.to_numeric(wkday_route_df['ROUTE_TOTAL'], errors='coerce')
wkday_route_df['ROUTE_TOTAL'].fillna(0, inplace=True)
wkend_route_df['ROUTE_TOTAL'] = pd.to_numeric(wkend_route_df['ROUTE_TOTAL'], errors='coerce')
wkend_route_df['ROUTE_TOTAL'].fillna(0, inplace=True)


In [13]:
test = pd.read_excel("VTA_CA_CR.xlsx", sheet_name=None)
test.keys()

dict_keys(['DATE_CHANGE', 'WkDAY-Overall', 'WkDAY-RouteTotal', 'WkDAY-RAIL', 'WkDAY-RailTotal', 'WkEND-Overall', 'WkEND-RouteTotal', 'WkEND-RAIL', 'WkEND-RailTotal', 'o2o-overall', 'o2o-RouteLevel', 'o2o-RAIL', 'o2o-RailTotal'])

In [14]:

df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )

In [15]:
df[['ROUTE_SURVEYEDCode_Splited','ROUTE_SURVEYEDCode']].head()

,ROUTE_SURVEYEDCode_Splited,ROUTE_SURVEYEDCode
0,UTA_1_1,UTA_1_1_01
1,UTA_1_1,UTA_1_1_01
2,UTA_1_209,UTA_1_209_01
3,UTA_1_200,UTA_1_200_00
4,UTA_1_200,UTA_1_200_01


In [16]:
df.shape

(14869, 741)

In [17]:
df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)

In [18]:
# wkday_overall_df

In [19]:
df=df[df['INTERV_INIT']!='999']
df=df[df['INTERV_INIT']!=999]

In [20]:
df.shape

(14790, 741)

In [21]:
df.drop_duplicates(subset='id',inplace=True)
df.shape

(14780, 741)

In [22]:
date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

In [23]:
def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)

In [24]:
def get_day_name(x):
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue
            
df['Day']=df['Date'].apply(get_day_name)

In [25]:
df.Day

31         Tuesday
33         Tuesday
54         Tuesday
55         Tuesday
58       Wednesday
           ...    
14863       Friday
14864       Friday
14865       Friday
14866       Friday
14867       Friday
Name: Day, Length: 14780, dtype: object

In [26]:
weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]

weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]

In [27]:
weekday_df.shape

(12945, 743)

In [29]:
weekend_df['ROUTE_SURVEYEDCode_Splited'].unique()

array(['UTA_1_704', 'UTA_1_703', 'UTA_1_551', 'UTA_1_47', 'UTA_1_205',
       'UTA_1_21', 'UTA_1_830X', 'UTA_1_470', 'UTA_1_2', 'UTA_1_750',
       'UTA_1_701', 'UTA_1_223', 'UTA_1_628', 'UTA_1_200', 'UTA_1_45',
       'UTA_1_640', 'UTA_1_1', 'UTA_1_201', 'UTA_1_240', 'UTA_1_509',
       'UTA_1_35', 'UTA_1_218', 'UTA_1_627', 'UTA_1_850', 'UTA_1_33',
       'UTA_1_626', 'UTA_1_667', 'UTA_1_720', 'UTA_1_39', 'UTA_1_871',
       'UTA_1_603X', 'UTA_1_612', 'UTA_1_606', 'UTA_1_220', 'UTA_1_209',
       'UTA_1_9', 'UTA_1_821', 'UTA_1_625', 'UTA_1_72', 'UTA_1_645',
       'UTA_1_62', 'UTA_1_4', 'UTA_1_604', 'UTA_1_217', 'UTA_1_213',
       'UTA_1_54', 'UTA_1_972'], dtype=object)

In [30]:
time_column_check=['timeoncode']
time_period_column_check=['timeon']
time_column=check_all_characters_present(df,time_column_check)
time_period_column=check_all_characters_present(df,time_period_column_check)
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)
stopon_clntid_column_check=['stoponclntid']
stopon_clntid_column=check_all_characters_present(df,stopon_clntid_column_check)
stopoff_clntid_column_check=['stopoffclntid']
stopoff_clntid_column=check_all_characters_present(df,stopon_clntid_column_check)

In [31]:
weekday_df[time_column[0]].dtype

dtype('float64')

In [32]:
weekday_df.shape,weekend_df.shape

((12945, 743), (1835, 743))

In [33]:
weekday_df.shape

(12945, 743)

In [35]:
weekday_df.dropna(subset=[time_column[0]],inplace=True)

In [44]:
weekday_df[[route_survey_column[0],'ROUTE_SURVEYED',stopon_clntid_column[0],stopoff_clntid_column[0],time_column[0],time_period_column[0],'Day','ELVIS_STATUS']]

,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,STOP_ON_CLNTID,STOP_ON_CLNTID,TIME_ONCode,TIME_ON,Day,ELVIS_STATUS
55,UTA_1_47_00,47 4700 SOUTH - TO W VALLEY CTL,UTA_1_47_00_14603,UTA_1_47_00_14603,16.0,7:30-8:30pm,Tuesday,NaN
59,UTA_1_47_00,47 4700 SOUTH - TO W VALLEY CTL,UTA_1_47_00_16077,UTA_1_47_00_16077,1.0,Before 6am,Wednesday,NaN
62,UTA_1_47_00,47 4700 SOUTH - TO W VALLEY CTL,UTA_1_47_00_16946,UTA_1_47_00_16946,1.0,Before 6am,Wednesday,NaN
63,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,UTA_1_750_01_21757,UTA_1_750_01_21757,1.0,Before 6am,Wednesday,NaN
66,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,UTA_1_750_01_21757,UTA_1_750_01_21757,1.0,Before 6am,Wednesday,NaN
...,...,...,...,...,...,...,...,...
14811,UTA_1_704_00,TRAX GREEN LINE 704 - TO WEST VALLEY,UTA_1_704_00_23050,UTA_1_704_00_23050,14.0,5:30-6:30pm,Thursday,NaN
14815,UTA_1_704_00,TRAX GREEN LINE 704 - TO WEST VALLEY,UTA_1_704_00_23050,UTA_1_704_00_23050,15.0,6:30-7:30pm,Thursday,NaN
14817,UTA_1_704_00,TRAX GREEN LINE 704 - TO WEST VALLEY,UTA_1_704_00_23050,UTA_1_704_00_23050,16.0,7:30-8:30pm,Thursday,NaN
14822,UTA_1_704_00,TRAX GREEN LINE 704 - TO WEST VALLEY,UTA_1_704_00_18414,UTA_1_704_00_18414,18.0,After 9:30pm,Thursday,NaN


In [45]:
weekend_df.dropna(subset=[time_column[0]],inplace=True)
weekend_df[[route_survey_column[0],'ROUTE_SURVEYED',stopon_clntid_column[0],stopoff_clntid_column[0],time_column[0],time_period_column[0],'Day','ELVIS_STATUS']]

,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,STOP_ON_CLNTID,STOP_ON_CLNTID,TIME_ONCode,TIME_ON,Day,ELVIS_STATUS
2579,UTA_1_704_01,TRAX GREEN LINE 704 - TO AIRPORT,UTA_1_704_01_22652,UTA_1_704_01_22652,2.0,6-7am,Saturday,NaN
2580,UTA_1_704_01,TRAX GREEN LINE 704 - TO AIRPORT,UTA_1_704_01_22648,UTA_1_704_01_22648,2.0,6-7am,Saturday,NaN
2581,UTA_1_703_01,TRAX RED LINE 703 - TO MEDICAL,UTA_1_703_01_18389,UTA_1_703_01_18389,2.0,6-7am,Saturday,NaN
2582,UTA_1_704_01,TRAX GREEN LINE 704 - TO AIRPORT,UTA_1_704_01_20982,UTA_1_704_01_20982,2.0,6-7am,Saturday,NaN
2583,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,UTA_1_703_00_18381,UTA_1_703_00_18381,3.0,7-8am,Saturday,NaN
...,...,...,...,...,...,...,...,...
14717,UTA_1_220_00,220 HIGHLAND DRIVE / 1300 EAST - TO SANDY,UTA_1_220_00_18194,UTA_1_220_00_18194,9.0,12:30-1:30pm,Saturday,NaN
14718,UTA_1_701_00,TRAX BLUE LINE 701 - TO DRAPER,UTA_1_701_00_21799,UTA_1_701_00_21799,9.0,12:30-1:30pm,Saturday,NaN
14719,UTA_1_701_00,TRAX BLUE LINE 701 - TO DRAPER,UTA_1_701_00_21799,UTA_1_701_00_21799,9.0,12:30-1:30pm,Saturday,NaN
14720,UTA_1_701_00,TRAX BLUE LINE 701 - TO DRAPER,UTA_1_701_00_21801,UTA_1_701_00_21801,8.0,11:30am-12:30pm,Saturday,NaN


In [30]:
wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)
wkday_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)

In [31]:
# def create_time_value_df_with_display(df):
#     """
#     Create a time-value DataFrame summarizing counts and time ranges.

#     Parameters:
#         df (pd.DataFrame): Input DataFrame containing the time values.
#         time_column (str): Name of the column in the input DataFrame containing the time values.

#     Returns:
#         pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
#     """
#     # Define time value groups
#     pre_early_am_values = [1]
#     early_am_values = [2]
#     am_values = [3, 4, 5, 6]
#     midday_values = [7, 8, 9, 10, 11]
#     pm_values = [12, 13, 14]
#     evening_values = [15, 16, 17, 18]

#     # Mapping time groups to corresponding columns
#     time_group_mapping = {
#         0: pre_early_am_values,
#         1: early_am_values,
#         2: am_values,
#         3: midday_values,
#         4: pm_values,
#         5: evening_values,
#     }

#     # Mapping time values to time ranges
#     time_mapping = {
#         1: 'Before 5:00 am',
#         2: '5:00 am - 6:00 am',
#         3: '6:00 am - 7:00 am',
#         4: '7:00 am - 8:00 am',
#         5: '8:00 am - 9:00 am',
#         6: '9:00 am - 10:00 am',
#         7: '10:00 am - 11:00 am',
#         8: '11:00 am - 12:00 pm',
#         9: '12:00 pm - 1:00 pm',
#         10: '1:00 pm - 2:00 pm',
#         11: '2:00 pm - 3:00 pm',
#         12: '3:00 pm - 4:00 pm',
#         13: '4:00 pm - 5:00 pm',
#         14: '5:00 pm - 6:00 pm',
#         15: '6:00 pm - 7:00 pm',
#         16: '7:00 pm - 8:00 pm',
#         17: '8:00 pm - 9:00 pm',
#         18: 'After 9:00 pm',
#     }

#     # Ensure the time_column is of integer type
#     df[time_column] = df[time_column].fillna(0).astype(int)

#     # Initialize the new DataFrame
#     new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

#     # Populate the DataFrame with counts
#     for col, values in time_group_mapping.items():
#         for value in values:
#             count = df[df[time_column[0]] == value].shape[0]
#             row = {"Original Text": value}

#             # Initialize all columns to 0
#             for c in range(6):
#                 row[c] = 0

#             # Update the corresponding column with the count
#             row[col] = count
#             new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

#     # Map time values to time ranges
#     new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

#     # Drop rows with missing time ranges
#     new_df.dropna(subset=['Time Range'], inplace=True)

#     # Add a display text column with sequential numbering
#     new_df['Display_Text'] = range(1, len(new_df) + 1)

#     return new_df


# # Usage
# wkend_time_value_df = create_time_value_df_with_display(weekend_df)
# wkday_time_value_df = create_time_value_df_with_display(weekday_df)


In [31]:
wkday_overall_df[[0,1,2,3,4,5]]=wkday_overall_df[[0,1,2,3,4,5]].fillna(0)
wkday_overall_df[[0,1,2,3,4,5]]

,0,1,2,3,4,5
0,0.0,0.0,3.86907,13.777178,27.183701,14.049301
1,0.0,0.0,1.218444,9.263885,11.780501,13.892867
2,0.0,0.0,0.699697,14.396686,15.461442,9.689579
3,0.0,0.0,1.731529,30.73615,29.856695,17.928494
4,0.0,0.0,2.608276,6.390914,4.120649,3.899369
...,...,...,...,...,...,...
171,0.0,0.0,5.430356,6.041328,5.843303,4.614533
172,0.0,0.0,5.886347,2.844195,2.20224,3.844613
173,0.0,0.0,1.090824,1.3455,1.221467,1.282619
174,0.0,0.0,2.075723,2.769709,2.515249,1.969445


In [32]:
wkend_overall_df[[0,1,2,3,4,5]]=wkend_overall_df[[0,1,2,3,4,5]].fillna(0)
wkend_overall_df[[0,1,2,3,4,5]]

,0,1,2,3,4,5
0,0.0,0.0,1,1,1,1
1,0.0,0.0,1,1,1,1
2,0.0,0.0,1,1,1,1
3,0.0,0.0,1,1,1,1
4,0.0,0.0,1,1,1,1
...,...,...,...,...,...,...
347,0.0,0.0,2,2,2,2
348,0.0,0.0,2,2,2,2
349,0.0,0.0,2,2,2,2
350,0.0,0.0,2,2,2,2


In [33]:
def create_route_direction_level_df(overalldf, df):
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    pre_early_am_column = [0]  # 0 is for Pre-Early AM header
    early_am_column = [1]  # 1 is for Early AM header
    am_column = [2]  # This is for AM header
    midday_column = [3]  # this is for MIDDAY header
    pm_column = [4]  # this is for PM header
    evening_column = [5]  # this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evening data and added values from Completion Report
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    new_df['STATION_ID']=overalldf['STATION_ID']
    new_df['STATION_ID_SPLITTED']=overalldf['STATION_ID_SPLITTED']
    new_df['CR_PRE_Early_AM'] = pd.to_numeric(overalldf[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Early_AM'] = pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overalldf[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)
#     new code added for merging the same ROUTE_SURVEYEDCode
#     new_df=new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
#     new_df.reset_index(drop=True, inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        station_id=row['STATION_ID_SPLITTED']

        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) &(df['STATION_ID_SPLITTED']==station_id) & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return new_df

In [34]:
wkend_route_direction_df=create_route_direction_level_df(wkend_overall_df,weekend_df)
wkend_route_direction_df

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,6.0,7.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,UTA_1_703_00,UTA_1_703_19169,19169,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,3.0,2.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,3.0,1.0,8.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,UTA_1_703_00,UTA_1_703_18384,18384,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,1.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,UTA_1_703_00,UTA_1_703_18381,18381,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,0.0,0.0,6.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,UTA_1_750_01,UTA_1_750_01_23074,23074,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,0.0,0.0,2.0,0.0,0.0,2.0,0.0,2.0,2.0,6.0
348,UTA_1_750_01,UTA_1_750_01_23071,23071,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0
349,UTA_1_750_01,UTA_1_750_01_25330,25330,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,0.0,0.0,1.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0
350,UTA_1_750_01,UTA_1_750_01_23076,23076,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,8.0


In [35]:
wkend_route_direction_df.describe()

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
count,352.0,352.0,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,352.000000,...,352.000000,352.000000,352.000000,352.0,352.0,352.000000,352.000000,352.000000,352.000000,352.000000
mean,0.0,0.0,1.500000,1.500000,1.500000,1.500000,6.000000,0.022727,0.079545,0.795455,...,0.625000,0.454545,3.920455,0.0,0.0,0.977273,0.536932,1.082386,1.190341,3.786932
std,0.0,0.0,0.500712,0.500712,0.500712,0.500712,2.002847,0.183494,0.291243,1.380855,...,1.167582,1.006068,3.972044,0.0,0.0,0.765752,0.727012,0.740776,0.720571,2.205290
min,0.0,0.0,1.000000,1.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.0,0.0,1.000000,1.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.0,0.0,0.000000,0.000000,1.000000,1.000000,2.000000
50%,0.0,0.0,1.500000,1.500000,1.500000,1.500000,6.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,3.000000,0.0,0.0,1.000000,0.000000,1.000000,1.000000,3.000000
75%,0.0,0.0,2.000000,2.000000,2.000000,2.000000,8.000000,0.000000,0.000000,1.000000,...,1.000000,1.000000,5.000000,0.0,0.0,2.000000,1.000000,2.000000,2.000000,6.000000
max,0.0,0.0,2.000000,2.000000,2.000000,2.000000,8.000000,2.000000,2.000000,9.000000,...,6.000000,7.000000,23.000000,0.0,0.0,2.000000,2.000000,2.000000,2.000000,8.000000


In [40]:
wkend_route_direction_df[wkend_route_direction_df['STATION_ID_SPLITTED']=='19167']
# wkend_route_direction_df[wkend_route_direction_df['ROUTE_SURVEYEDCode']=='UTA_1_703_01']

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,1.0,1.0,1.0,4.0,...,3.0,1.0,8.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
178,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,3.0,1.0,8.0,0.0,0.0,2.0,0.0,0.0,1.0,3.0


In [38]:
# wkday_route_direction_df=create_route_direction_level_df(wkday_overall_df,weekday_df)
# wkday_route_direction_df

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,4.0,14.0,28.0,15.0,61.0,...,31.0,13.0,88.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0
1,UTA_1_703_00,UTA_1_703_19169,19169,0.0,0.0,2.0,10.0,12.0,14.0,38.0,...,20.0,14.0,45.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,15.0,16.0,10.0,42.0,...,15.0,10.0,47.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,UTA_1_703_00,UTA_1_703_18384,18384,0.0,0.0,2.0,31.0,30.0,18.0,81.0,...,28.0,35.0,92.0,0.0,0.0,0.0,6.0,2.0,0.0,8.0
4,UTA_1_703_00,UTA_1_703_18381,18381,0.0,0.0,3.0,7.0,5.0,4.0,19.0,...,5.0,4.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,UTA_1_750_01,UTA_1_750_01_23074,23074,0.0,0.0,6.0,7.0,6.0,5.0,24.0,...,12.0,4.0,34.0,0.0,0.0,3.0,0.0,0.0,1.0,4.0
172,UTA_1_750_01,UTA_1_750_01_23071,23071,0.0,0.0,6.0,3.0,3.0,4.0,16.0,...,6.0,5.0,20.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0
173,UTA_1_750_01,UTA_1_750_01_25330,25330,0.0,0.0,2.0,2.0,2.0,2.0,8.0,...,1.0,1.0,5.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0
174,UTA_1_750_01,UTA_1_750_01_23076,23076,0.0,0.0,3.0,3.0,3.0,2.0,11.0,...,6.0,3.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
pre_early_am_values = [1]
early_am_values = [2]
am_values = [3, 4, 5, 6]
midday_values = [7, 8, 9, 10, 11]
pm_values = [12, 13, 14]
evening_values = [15, 16, 17, 18]

pre_early_am_column=[0]  #0 is for Pre-Early AM header
early_am_column=[1]  #1 is for Early AM header
am_column=[2] #This is for AM header
midday_column=[3] #this is for MIDDAY header
pm_column=[4] #this is for PM header
evening_column=[5] #this is for EVENING header

def convert_string_to_integer(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return 0

# Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
new_df=pd.DataFrame()
new_df['ROUTE_SURVEYEDCode']=wkend_overall_df['LS_NAME_CODE']
new_df['STATION_ID']=wkend_overall_df['STATION_ID']
new_df['STATION_ID_SPLITTED']=wkend_overall_df['STATION_ID_SPLITTED']

In [69]:
new_df[new_df['STATION_ID_SPLITTED']=='19171']

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED
0,UTA_1_703_00,UTA_1_703_19171,19171
176,UTA_1_703_00,UTA_1_703_19171,19171


In [70]:
new_df['CR_PRE_Early_AM'] = pd.to_numeric(wkend_overall_df[pre_early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df['CR_Early_AM'] = pd.to_numeric(wkend_overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df['CR_AM_Peak'] = pd.to_numeric(wkend_overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df['CR_Midday'] = pd.to_numeric(wkend_overall_df[midday_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df['CR_PM_Peak'] = pd.to_numeric(wkend_overall_df[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df['CR_Evening'] = pd.to_numeric(wkend_overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
new_df.fillna(0,inplace=True)

In [92]:
# new_df = new_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()



In [93]:
# Step 5: Reset index (optional, already done with as_index=False)
# new_df.reset_index(drop=True, inplace=True)

In [71]:
new_df['ROUTE_SURVEYEDCode_Splitted']=new_df['ROUTE_SURVEYEDCode'].apply(edit_ls_code_column)

In [72]:
new_df[['ROUTE_SURVEYEDCode_Splitted','ROUTE_SURVEYEDCode']].head()

,ROUTE_SURVEYEDCode_Splitted,ROUTE_SURVEYEDCode
0,UTA_1_703,UTA_1_703_00
1,UTA_1_703,UTA_1_703_00
2,UTA_1_703,UTA_1_703_00
3,UTA_1_703,UTA_1_703_00
4,UTA_1_703,UTA_1_703_00


In [73]:
new_df[(new_df['STATION_ID_SPLITTED']=='19171')&(new_df['ROUTE_SURVEYEDCode_Splitted']=='UTA_1_703')]

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,ROUTE_SURVEYEDCode_Splitted
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703
176,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_703


In [75]:
filtered_df = new_df[(new_df['STATION_ID_SPLITTED'] == '19171') & 
                     (new_df['ROUTE_SURVEYEDCode_Splitted'] == 'UTA_1_703')]

# Sum the rows and return as a single row
summed_row = filtered_df.sum(numeric_only=True).to_frame().T
summed_row.reset_index(drop=True, inplace=True)
summed_row

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening
0,0.0,0.0,3.0,3.0,3.0,3.0


In [54]:
new_df.head()

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,ROUTE_SURVEYEDCode_Splitted
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703
1,UTA_1_703_00,UTA_1_703_19169,19169,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703
3,UTA_1_703_00,UTA_1_703_18384,18384,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703
4,UTA_1_703_00,UTA_1_703_18381,18381,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703


In [96]:
wkend_overall_df['LS_NAME_CODE'].unique()

array(['UTA_1_703_00', 'UTA_1_703_01', 'UTA_1_701_00', 'UTA_1_701_01',
       'UTA_1_704_00', 'UTA_1_704_01', 'UTA_1_750_00', 'UTA_1_750_01'],
      dtype=object)

In [72]:
new_df['ROUTE_SURVEYEDCode'].unique()

array(['UTA_1_701_00', 'UTA_1_701_01', 'UTA_1_703_00', 'UTA_1_703_01',
       'UTA_1_704_00', 'UTA_1_704_01', 'UTA_1_750_00', 'UTA_1_750_01'],
      dtype=object)

# Station Wise Route Level Comparison

In [76]:

for index, row in new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode_Splitted']
    station_id=row['STATION_ID_SPLITTED']
    def get_counts_and_ids(time_values):
        # Just for SALEM
        # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
        subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code)& (df['STATION_ID_SPLITTED']==station_id)& (df[time_column[0]].isin(time_values))]
        subset_df=subset_df.drop_duplicates(subset='id')
        count = subset_df.shape[0]
        ids = subset_df['id'].values
        return count, ids

    pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
    early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
#     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
    new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
    # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
    new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
    new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value


In [78]:
new_df['ROUTE_SURVEYEDCode_Splitted'].unique()

array(['UTA_1_703', 'UTA_1_701', 'UTA_1_704', 'UTA_1_750'], dtype=object)

In [81]:
unique_station_ids=new_df['STATION_ID_SPLITTED'].unique()
unique_station_ids

array(['19171', '19169', '19167', '18384', '18381', '18379', '18377',
       '18410', '25329', '20973', '18412', '18414', '18388', '18390',
       '18392', '18394', '18396', '24144', '22636', '22634', '22632',
       '22630', '22628', '22626', '22624', '22622', '22639', '22620',
       '22621', '22623', '22625', '22627', '22629', '22631', '22633',
       '22635', '18395', '18393', '18391', '18389', '18387', '18413',
       '18411', '20982', '25328', '18409', '18378', '18380', '18382',
       '18383', '19166', '19168', '19170', '21799', '21801', '21762',
       '18406', '18408', '18386', '18376', '18398', '18400', '18402',
       '21226', '18404', '23319', '23321', '23323', '23322', '23320',
       '23318', '18403', '21225', '18401', '18399', '18397', '18415',
       '18385', '18407', '18405', '21800', '21763', '21761', '23050',
       '23048', '23046', '23044', '23042', '23040', '23092', '22645',
       '22647', '22649', '22651', '22652', '22650', '22648', '22646',
       '23039', '230

In [82]:
new_df

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,ROUTE_SURVEYEDCode_Splitted,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,0.0,4.0,18.0,32.0,37.0,20.0,111.0
1,UTA_1_703_00,UTA_1_703_19169,19169,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,1.0,0.0,2.0,15.0,23.0,16.0,57.0
2,UTA_1_703_00,UTA_1_703_19167,19167,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,0.0,1.0,4.0,21.0,18.0,11.0,55.0
3,UTA_1_703_00,UTA_1_703_18384,18384,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,0.0,0.0,5.0,26.0,29.0,36.0,96.0
4,UTA_1_703_00,UTA_1_703_18381,18381,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,0.0,0.0,11.0,12.0,5.0,4.0,32.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,UTA_1_750_01,UTA_1_750_01_23074,23074,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_750,8.0,6.0,0.0,10.0,32.0,29.0,13.0,90.0
348,UTA_1_750_01,UTA_1_750_01_23071,23071,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_750,8.0,3.0,1.0,13.0,8.0,20.0,11.0,56.0
349,UTA_1_750_01,UTA_1_750_01_25330,25330,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_750,8.0,2.0,1.0,7.0,9.0,6.0,6.0,31.0
350,UTA_1_750_01,UTA_1_750_01_23076,23076,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_750,8.0,4.0,0.0,13.0,24.0,38.0,20.0,99.0


In [85]:
# Initialize an empty list to store results
results = []

# Iterate over unique station IDs
for station_id in unique_station_ids:
    # Filter DataFrame for the current station ID
    station_df = new_df[new_df['STATION_ID_SPLITTED'] == station_id]
    
    # Iterate over unique ROUTE_SURVEYEDCode_Splitted for the current station ID
    for route_code in station_df['ROUTE_SURVEYEDCode_Splitted'].unique():
        # Filter rows for the specific route and station
        filtered_df = station_df[station_df['ROUTE_SURVEYEDCode_Splitted'] == route_code]
        
        # Sum numeric columns and convert to a single row
        summed_row = filtered_df.sum(numeric_only=True).to_frame().T
        
        # Add key identifying columns
        summed_row['ROUTE_SURVEYEDCode'] = station_df.iloc[0]['ROUTE_SURVEYEDCode']
        summed_row['STATION_ID'] = station_df.iloc[0]['STATION_ID']
        summed_row['STATION_ID_SPLITTED'] = station_id
        summed_row['ROUTE_SURVEYEDCode_Splitted'] = route_code
        
        # Append the row to results
        results.append(summed_row)

# Concatenate all results into a new DataFrame
route_station_wise = pd.concat(results, ignore_index=True)

# Display the resulting DataFrame
route_station_wise


,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,ROUTE_SURVEYEDCode_Splitted
0,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,8.0,36.0,64.0,74.0,40.0,222.0,UTA_1_703_00,UTA_1_703_19171,19171,UTA_1_703
1,0.0,0.0,3.0,3.0,3.0,3.0,12.0,2.0,0.0,4.0,30.0,46.0,32.0,114.0,UTA_1_703_00,UTA_1_703_19169,19169,UTA_1_703
2,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,2.0,8.0,42.0,36.0,22.0,110.0,UTA_1_703_00,UTA_1_703_19167,19167,UTA_1_703
3,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,10.0,52.0,58.0,72.0,192.0,UTA_1_703_00,UTA_1_703_18384,18384,UTA_1_703
4,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,22.0,24.0,10.0,8.0,64.0,UTA_1_703_00,UTA_1_703_18381,18381,UTA_1_703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,0.0,0.0,6.0,6.0,6.0,6.0,24.0,4.0,4.0,32.0,40.0,24.0,40.0,144.0,UTA_1_750_00,UTA_1_750_21758,21758,UTA_1_750
155,0.0,0.0,6.0,6.0,6.0,6.0,24.0,28.0,24.0,32.0,36.0,44.0,40.0,204.0,UTA_1_750_00,UTA_1_750_21772,21772,UTA_1_750
156,0.0,0.0,6.0,6.0,6.0,6.0,24.0,48.0,32.0,72.0,52.0,24.0,44.0,272.0,UTA_1_750_00,UTA_1_750_21757,21757,UTA_1_750
157,0.0,0.0,6.0,6.0,6.0,6.0,24.0,68.0,28.0,36.0,32.0,40.0,16.0,220.0,UTA_1_750_00,UTA_1_750_21760,21760,UTA_1_750


In [91]:
for index, row in route_station_wise.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
#         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        route_station_wise.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        route_station_wise.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_station_wise.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_station_wise.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_station_wise.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_station_wise.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        route_station_wise.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))


In [92]:
route_station_wise

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,...,STATION_ID,STATION_ID_SPLITTED,ROUTE_SURVEYEDCode_Splitted,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,8.0,36.0,...,UTA_1_703_19171,19171,UTA_1_703,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,3.0,3.0,3.0,3.0,12.0,2.0,0.0,4.0,...,UTA_1_703_19169,19169,UTA_1_703,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,2.0,8.0,...,UTA_1_703_19167,19167,UTA_1_703,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,10.0,...,UTA_1_703_18384,18384,UTA_1_703,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,22.0,...,UTA_1_703_18381,18381,UTA_1_703,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,0.0,0.0,6.0,6.0,6.0,6.0,24.0,4.0,4.0,32.0,...,UTA_1_750_21758,21758,UTA_1_750,0.0,0.0,0.0,0.0,0.0,0.0,0.0
155,0.0,0.0,6.0,6.0,6.0,6.0,24.0,28.0,24.0,32.0,...,UTA_1_750_21772,21772,UTA_1_750,0.0,0.0,0.0,0.0,0.0,0.0,0.0
156,0.0,0.0,6.0,6.0,6.0,6.0,24.0,48.0,32.0,72.0,...,UTA_1_750_21757,21757,UTA_1_750,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157,0.0,0.0,6.0,6.0,6.0,6.0,24.0,68.0,28.0,36.0,...,UTA_1_750_21760,21760,UTA_1_750,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [95]:
route_station_wise.drop(columns=['ROUTE_SURVEYEDCode_Splitted','STATION_ID_SPLITTED']).head()

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,...,STATION_ID,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,ROUTE_SURVEYED,STATION_NAME
0,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,8.0,36.0,...,UTA_1_703_19171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TRAX RED LINE 703 - TO DAYBREAK,University Medical Center
1,0.0,0.0,3.0,3.0,3.0,3.0,12.0,2.0,0.0,4.0,...,UTA_1_703_19169,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TRAX RED LINE 703 - TO DAYBREAK,Fort Douglas Station
2,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,2.0,8.0,...,UTA_1_703_19167,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TRAX RED LINE 703 - TO DAYBREAK,University South Campus Station
3,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,10.0,...,UTA_1_703_18384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TRAX RED LINE 703 - TO DAYBREAK,Stadium Station
4,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,0.0,22.0,...,UTA_1_703_18381,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TRAX RED LINE 703 - TO DAYBREAK,900 East Station


In [86]:
route_station_wise.columns

Index(['CR_PRE_Early_AM', 'CR_Early_AM', 'CR_AM_Peak', 'CR_Midday',
       'CR_PM_Peak', 'CR_Evening', 'CR_Total', 'DB_PRE_Early_AM_Peak',
       'DB_Early_AM_Peak', 'DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak',
       'DB_Evening', 'DB_Total', 'ROUTE_SURVEYEDCode', 'STATION_ID',
       'STATION_ID_SPLITTED', 'ROUTE_SURVEYEDCode_Splitted'],
      dtype='object')

In [93]:
for _,row in route_station_wise.iterrows():
    route_surveyed=detail_df_stops[detail_df_stops['ETC_ROUTE_ID']==row['ROUTE_SURVEYEDCode']]['ETC_ROUTE_NAME'].iloc[0]
    route_surveyed_ID=detail_df_stops[detail_df_stops['ETC_ROUTE_ID']==row['ROUTE_SURVEYEDCode']]['ETC_ROUTE_ID'].iloc[0]
    station_name=wkday_overall_df[wkday_overall_df['STATION_ID']==row['STATION_ID']]['STATION_NAME'].iloc[0]
    
    route_station_wise.loc[row.name,'ROUTE_SURVEYED']=route_surveyed        
    route_station_wise.loc[row.name,'STATION_NAME']=station_name    

# STATION WISE ROUTE LEVEL CODE ENDS HERE

In [90]:
len('WkEND Stationwise Comparison')

28

In [83]:
new_df.columns

Index(['ROUTE_SURVEYEDCode', 'STATION_ID', 'STATION_ID_SPLITTED',
       'CR_PRE_Early_AM', 'CR_Early_AM', 'CR_AM_Peak', 'CR_Midday',
       'CR_PM_Peak', 'CR_Evening', 'ROUTE_SURVEYEDCode_Splitted', 'CR_Total',
       'DB_PRE_Early_AM_Peak', 'DB_Early_AM_Peak', 'DB_AM_Peak', 'DB_Midday',
       'DB_PM_Peak', 'DB_Evening', 'DB_Total'],
      dtype='object')

In [ ]:
for station_id in unique_station_ids:
    print()

In [79]:
filtered_df = new_df[(new_df['STATION_ID_SPLITTED'] == '19171') & 
                     (new_df['ROUTE_SURVEYEDCode_Splitted'] == 'UTA_1_703')]

# Sum the rows and return as a single row
summed_row = filtered_df.sum(numeric_only=True).to_frame().T
summed_row.reset_index(drop=True, inplace=True)
summed_row

,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total
0,0.0,0.0,3.0,3.0,3.0,3.0,12.0,0.0,8.0,36.0,64.0,74.0,40.0,222.0


In [77]:
new_df[(new_df['STATION_ID_SPLITTED']=='19171')&(new_df['ROUTE_SURVEYEDCode_Splitted']=='UTA_1_703')]

,ROUTE_SURVEYEDCode,STATION_ID,STATION_ID_SPLITTED,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,ROUTE_SURVEYEDCode_Splitted,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total
0,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,1.0,1.0,1.0,1.0,UTA_1_703,4.0,0.0,4.0,18.0,32.0,37.0,20.0,111.0
176,UTA_1_703_00,UTA_1_703_19171,19171,0.0,0.0,2.0,2.0,2.0,2.0,UTA_1_703,8.0,0.0,4.0,18.0,32.0,37.0,20.0,111.0


# SImple Route LEVEL COMPARISON

In [97]:

# for index, row in new_df.iterrows():
#     route_code = row['ROUTE_SURVEYEDCode']

#     def get_counts_and_ids(time_values):
#         # Just for SALEM
#         # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
#         subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
#         subset_df=subset_df.drop_duplicates(subset='id')
#         count = subset_df.shape[0]
#         ids = subset_df['id'].values
#         return count, ids

#     pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
#     early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
#     am_value, am_value_ids = get_counts_and_ids(am_values)
#     midday_value, midday_value_ids = get_counts_and_ids(midday_values)
#     pm_value, pm_value_ids = get_counts_and_ids(pm_values)
#     evening_value, evening_value_ids = get_counts_and_ids(evening_values)
# #     print(pre_early_am_value,early_am_value,am_value,midday_value,pm_value,evening_value)
#     new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
#     new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
#     # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
#     new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
#     new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
#     new_df.loc[index, 'DB_AM_Peak'] = am_value
#     new_df.loc[index, 'DB_Midday'] = midday_value
#     new_df.loc[index, 'DB_PM_Peak'] = pm_value
#     new_df.loc[index, 'DB_Evening'] = evening_value
#     new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value


In [99]:
new_df

,ROUTE_SURVEYEDCode,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total
0,UTA_1_701_00,0.0,0.0,75.0,75.0,75.0,75.0,300.0,7.0,16.0,78.0,178.0,115.0,91.0,485.0
1,UTA_1_701_01,0.0,0.0,75.0,75.0,75.0,75.0,300.0,15.0,22.0,83.0,154.0,102.0,76.0,452.0
2,UTA_1_703_00,0.0,0.0,81.0,81.0,81.0,81.0,324.0,6.0,24.0,103.0,235.0,206.0,160.0,734.0
3,UTA_1_703_01,0.0,0.0,78.0,78.0,78.0,78.0,312.0,18.0,28.0,166.0,274.0,179.0,116.0,781.0
4,UTA_1_704_00,0.0,0.0,60.0,60.0,60.0,60.0,240.0,3.0,24.0,124.0,185.0,130.0,78.0,544.0
5,UTA_1_704_01,0.0,0.0,57.0,57.0,57.0,57.0,228.0,24.0,23.0,131.0,142.0,122.0,66.0,508.0
6,UTA_1_750_00,0.0,0.0,51.0,51.0,51.0,51.0,204.0,20.0,11.0,101.0,214.0,168.0,110.0,624.0
7,UTA_1_750_01,0.0,0.0,51.0,51.0,51.0,51.0,204.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0


In [75]:
new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
new_df

,ROUTE_SURVEYEDCode,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,ROUTE_SURVEYEDCode_Splited
0,UTA_1_701_00,0.0,0.0,75.0,75.0,75.0,75.0,300.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_701
1,UTA_1_701_01,0.0,0.0,75.0,75.0,75.0,75.0,300.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_701
2,UTA_1_703_00,0.0,0.0,81.0,81.0,81.0,81.0,324.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_703
3,UTA_1_703_01,0.0,0.0,78.0,78.0,78.0,78.0,312.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_703
4,UTA_1_704_00,0.0,0.0,60.0,60.0,60.0,60.0,240.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_704
5,UTA_1_704_01,0.0,0.0,57.0,57.0,57.0,57.0,228.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_704
6,UTA_1_750_00,0.0,0.0,51.0,51.0,51.0,51.0,204.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_750
7,UTA_1_750_01,0.0,0.0,51.0,51.0,51.0,51.0,204.0,71.0,48.0,128.0,165.0,160.0,129.0,701.0,UTA_1_750


In [50]:
wkday_route_df.columns

Index(['SORT', 'LS_NAME_CODE', 'LS_NAME', 'ENTRY STATION', 'ETC_ROUTE_ID',
       'ETC_ROUTE_NAME', 'SINGLE ENTRY STATION', 'ROUTE_TOTAL'],
      dtype='object')

In [51]:
wkend_route_df.columns

Index(['DAY', 'SORT', 'LS_NAME_CODE', 'LS_NAME', 'ENTRY STATION',
       'ETC_ROUTE_ID', 'ETC_ROUTE_NAME', 'SINGLE ENTRY STATION',
       'ROUTE_TOTAL'],
      dtype='object')

In [63]:
route_level_df=pd.DataFrame()

unique_routes=new_df['ROUTE_SURVEYEDCode_Splitted'].unique()


unique_routes

array(['UTA_1_703', 'UTA_1_701', 'UTA_1_704', 'UTA_1_750'], dtype=object)

In [64]:
route_level_df['ROUTE_SURVEYEDCode']=unique_routes

wkend_route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','ETC_ROUTE_ID':'ROUTE_SURVEYEDCode'},inplace=True)

In [65]:
route_level_df

,ROUTE_SURVEYEDCode
0,UTA_1_703
1,UTA_1_701
2,UTA_1_704
3,UTA_1_750


In [78]:
wkday_route_df

,SORT,LS_NAME_CODE,LS_NAME,ENTRY STATION,ETC_ROUTE_ID,ETC_ROUTE_NAME,SINGLE ENTRY STATION,ROUTE_TOTAL
0,1,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,TRAX RED LINE 703 - TO DAYBREAK: University Me...,UTA_1_703,TRAX RED LINE 703,TRAX RED LINE 703: University Medical Center,88.318875
1,2,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,TRAX RED LINE 703 - TO DAYBREAK: Fort Douglas ...,UTA_1_703,TRAX RED LINE 703,TRAX RED LINE 703: Fort Douglas Station,0.000000
2,3,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,TRAX RED LINE 703 - TO DAYBREAK: University So...,UTA_1_703,TRAX RED LINE 703,TRAX RED LINE 703: University South Campus Sta...,0.000000
3,4,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,TRAX RED LINE 703 - TO DAYBREAK: Stadium Station,UTA_1_703,TRAX RED LINE 703,TRAX RED LINE 703: Stadium Station,0.000000
4,5,UTA_1_703_00,TRAX RED LINE 703 - TO DAYBREAK,TRAX RED LINE 703 - TO DAYBREAK: 900 East Station,UTA_1_703,TRAX RED LINE 703,TRAX RED LINE 703: 900 East Station,0.000000
...,...,...,...,...,...,...,...,...
171,172,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,FRONTRUNNER 750 - SOUTHBOUND: Lehi Station,UTA_1_750,FRONTRUNNER 750,FRONTRUNNER 750: Lehi Station,0.000000
172,173,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,FRONTRUNNER 750 - SOUTHBOUND: American Fork St...,UTA_1_750,FRONTRUNNER 750,FRONTRUNNER 750: American Fork Station,0.000000
173,174,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,FRONTRUNNER 750 - SOUTHBOUND: Vineyard Station,UTA_1_750,FRONTRUNNER 750,FRONTRUNNER 750: Vineyard Station,0.000000
174,175,UTA_1_750_01,FRONTRUNNER 750 - SOUTHBOUND,FRONTRUNNER 750 - SOUTHBOUND: Orem Central Sta...,UTA_1_750,FRONTRUNNER 750,FRONTRUNNER 750: Orem Central Station,0.000000


In [79]:
route_level_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   ROUTE_SURVEYEDCode  4 non-null      object
dtypes: object(1)
memory usage: 160.0+ bytes


In [80]:
for col in route_level_df.columns:
    if col != 'ROUTE_SURVEYEDCode':  # Skip grouping column
        route_level_df[col] = pd.to_numeric(route_level_df[col], errors='coerce')

In [81]:
route_level_df

,ROUTE_SURVEYEDCode
0,UTA_1_701
1,UTA_1_703
2,UTA_1_704
3,UTA_1_750


In [66]:
wkend_route_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)
route_level_df=pd.merge(route_level_df,wkend_route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']],on='ROUTE_SURVEYEDCode')
route_level_df=route_level_df.groupby('ROUTE_SURVEYEDCode', as_index=False).sum()
route_level_df.reset_index(drop=True, inplace=True)

In [67]:
route_level_df

,ROUTE_SURVEYEDCode,CR_Overall_Goal
0,UTA_1_701,1500
1,UTA_1_703,1590
2,UTA_1_704,1170
3,UTA_1_750,1020


In [84]:
wkend_route_df[wkend_route_df['ROUTE_SURVEYEDCode']=='UTA_1_750']['CR_Overall_Goal'].sum()

1020

In [85]:
for index , row in route_level_df.iterrows():
    subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
    # sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()



    sum_per_route_cr = subset_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    # sum_per_route_cr = subset_df[['CR_EARLY_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    sum_per_route_db = subset_df[['DB_PRE_Early_AM_Peak','DB_Early_AM_Peak','DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()

    route_level_df.loc[index,'CR_PRE_Early_AM']=sum_per_route_cr['CR_PRE_Early_AM']
    route_level_df.loc[index,'CR_Early_AM']=sum_per_route_cr['CR_Early_AM']
    route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
    route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
    route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
    route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
    route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
    
    route_level_df.loc[index,'DB_PRE_Early_AM_Peak']=sum_per_route_db['DB_PRE_Early_AM_Peak']
    route_level_df.loc[index,'DB_Early_AM_Peak']=sum_per_route_db['DB_Early_AM_Peak']
    route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
    route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
    route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
    route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
    route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   

In [86]:
route_level_df

,ROUTE_SURVEYEDCode,CR_Overall_Goal,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total
0,UTA_1_701,1500,0.0,0.0,150.0,150.0,150.0,150.0,600.0,142.0,96.0,256.0,330.0,320.0,258.0,1402.0
1,UTA_1_703,1590,0.0,0.0,159.0,159.0,159.0,159.0,636.0,142.0,96.0,256.0,330.0,320.0,258.0,1402.0
2,UTA_1_704,1170,0.0,0.0,117.0,117.0,117.0,117.0,468.0,142.0,96.0,256.0,330.0,320.0,258.0,1402.0
3,UTA_1_750,1020,0.0,0.0,102.0,102.0,102.0,102.0,408.0,142.0,96.0,256.0,330.0,320.0,258.0,1402.0


In [87]:
for index, row in route_level_df.iterrows():
    pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
    early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
    am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
    midday_diff=row['CR_Midday']-row['DB_Midday']    
    pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
    evening_diff=row['CR_Evening']-row['DB_Evening']
    total_diff=row['CR_Total']-row['DB_Total']
    overall_difference=row['CR_Overall_Goal']-row['DB_Total']
    route_level_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
    route_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
    route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
    route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
    route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
    route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
    # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
    route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
    route_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))

In [89]:
route_level_df.columns

Index(['ROUTE_SURVEYEDCode', 'CR_Overall_Goal', 'CR_PRE_Early_AM',
       'CR_Early_AM', 'CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening',
       'CR_Total', 'DB_PRE_Early_AM_Peak', 'DB_Early_AM_Peak', 'DB_AM_Peak',
       'DB_Midday', 'DB_PM_Peak', 'DB_Evening', 'DB_Total',
       'PRE_Early_AM_DIFFERENCE', 'Early_AM_DIFFERENCE', 'AM_DIFFERENCE',
       'Midday_DIFFERENCE', 'PM_DIFFERENCE', 'Evening_DIFFERENCE',
       'Total_DIFFERENCE', 'Overall_Goal_DIFFERENCE'],
      dtype='object')

In [90]:
route_level_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal','DB_Total','Overall_Goal_DIFFERENCE']]

,ROUTE_SURVEYEDCode,CR_Overall_Goal,DB_Total,Overall_Goal_DIFFERENCE
0,UTA_1_701,1500,1402.0,98.0
1,UTA_1_703,1590,1402.0,188.0
2,UTA_1_704,1170,1402.0,0.0
3,UTA_1_750,1020,1402.0,0.0


In [ ]:
def create_time_value_df_with_display(overall_df, df):
    """
    Create a time-value DataFrame summarizing counts and time ranges, considering only records where 
    'ROUTE_SURVEYEDCode' from overall_df matches 'ROUTE_SURVEYEDCode_Splited' in df.

    Parameters:
        overall_df (pd.DataFrame): DataFrame with the column 'ROUTE_SURVEYEDCode'.
        df (pd.DataFrame): Input DataFrame containing the time values and 'ROUTE_SURVEYEDCode_Splited'.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # Filter df where overall_df['ROUTE_SURVEYEDCode'] == df['ROUTE_SURVEYEDCode_Splited']
    matched_df = df[df['ROUTE_SURVEYEDCode_Splited'].isin(overall_df['ROUTE_SURVEYEDCode'])]

    # Define time value groups
    pre_early_am_values = [1]
    early_am_values = [2]
    am_values = [3, 4, 5, 6]
    midday_values = [7, 8, 9, 10, 11]
    pm_values = [12, 13, 14]
    evening_values = [15, 16, 17, 18]

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        0: pre_early_am_values,
        1: early_am_values,
        2: am_values,
        3: midday_values,
        4: pm_values,
        5: evening_values,
    }

    # Mapping time values to time ranges
    time_mapping = {
        1: 'Before 5:00 am',
        2: '5:00 am - 6:00 am',
        3: '6:00 am - 7:00 am',
        4: '7:00 am - 8:00 am',
        5: '8:00 am - 9:00 am',
        6: '9:00 am - 10:00 am',
        7: '10:00 am - 11:00 am',
        8: '11:00 am - 12:00 pm',
        9: '12:00 pm - 1:00 pm',
        10: '1:00 pm - 2:00 pm',
        11: '2:00 pm - 3:00 pm',
        12: '3:00 pm - 4:00 pm',
        13: '4:00 pm - 5:00 pm',
        14: '5:00 pm - 6:00 pm',
        15: '6:00 pm - 7:00 pm',
        16: '7:00 pm - 8:00 pm',
        17: '8:00 pm - 9:00 pm',
        18: 'After 9:00 pm',
    }

    # Ensure the time_column is of integer type
    matched_df[time_column] = matched_df[time_column].fillna(0).astype(int)

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4, 5])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = matched_df[matched_df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df

In [ ]:
create_time_value_df_with_display(route_level_df,weekend_df)